In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pinn import PINN, SIREN, SafeNet, pde_loss, bc_loss, u_exact

In [2]:
def sample_interior(N):
    pts = torch.rand(N, 2)
    pts.requires_grad_(True)
    return pts


def sample_boundary(N):
    assert N % 4 == 0, f"N must be divisible by 4, got {N}"
    n = N // 4
    # bottom: y=0, x in [0,1]
    bottom = torch.stack([torch.rand(n), torch.zeros(n)], dim=1)
    # top: y=1
    top = torch.stack([torch.rand(n), torch.ones(n)], dim=1)
    # left: x=0
    left = torch.stack([torch.zeros(n), torch.rand(n)], dim=1)
    # right: x=1
    right = torch.stack([torch.ones(n), torch.rand(n)], dim=1)
    return torch.cat([bottom, top, left, right], dim=0)

In [3]:
N_test = 100
xy_int = sample_interior(N_test)
xy_bc = sample_boundary(N_test)
assert xy_int.shape == (N_test, 2) and xy_int.requires_grad
assert xy_bc.shape == (N_test, 2)
# All boundary points should be on edge
assert ((xy_bc[:, 0] == 0) | (xy_bc[:, 0] == 1) |
        (xy_bc[:, 1] == 0) | (xy_bc[:, 1] == 1)).all()
print("Семплирование: ОК")

Семплирование: ОК


In [4]:
import optuna
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS       = 40
N_EPOCHS_TRIAL = 2000
PRUNE_INTERVAL = 400
N_VIS_OPT      = 60

# Дефолты на случай если пропустить эту ячейку
best_pinn    = {}
best_siren   = {}
best_safenet = {}


def _eval_grid():
    x = torch.linspace(0, 1, N_VIS_OPT, device=device)
    y = torch.linspace(0, 1, N_VIS_OPT, device=device)
    XX, YY = torch.meshgrid(x, y, indexing='ij')
    xy = torch.stack([XX.flatten(), YY.flatten()], dim=1)
    u_t = u_exact(xy[:, 0], xy[:, 1]).reshape(N_VIS_OPT, N_VIS_OPT).cpu().numpy()
    return xy, u_t

_xy_vis_opt, _u_true_opt = _eval_grid()


def _rel_l2_opt(model):
    model.eval()
    with torch.no_grad():
        u_p = model(_xy_vis_opt).squeeze().reshape(N_VIS_OPT, N_VIS_OPT).cpu().numpy()
    model.train()
    return float(np.linalg.norm(u_p - _u_true_opt) / np.linalg.norm(_u_true_opt))


def _run_study(build_model_fn, n_trials=N_TRIALS, name='study'):
    def objective(trial):
        model, lam = build_model_fn(trial)
        lr  = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        xy_col = torch.rand(2000, 2, device=device, requires_grad=True)
        xy_bc  = sample_boundary(500).to(device)
        for ep in range(1, N_EPOCHS_TRIAL + 1):
            xy_col.grad = None
            opt.zero_grad()
            loss = pde_loss(model, xy_col) + lam * bc_loss(model, xy_bc)
            if not torch.isfinite(loss):
                return 1.0
            loss.backward()
            opt.step()
            if ep % PRUNE_INTERVAL == 0:
                l2 = _rel_l2_opt(model)
                trial.report(l2, ep)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
        return _rel_l2_opt(model)

    study = optuna.create_study(
        direction='minimize',
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=PRUNE_INTERVAL * 2),
        study_name=name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"{name}: лучшие параметры = {study.best_params}, L2 = {study.best_value:.4e}")
    return study.best_params


# --- PINN ---
print("=== Поиск гиперпараметров PINN ===")
def _build_pinn(trial):
    width  = trial.suggest_categorical('hidden_width',  [32, 64, 128])
    layers = trial.suggest_int('hidden_layers', 2, 6)
    lam    = trial.suggest_float('lambda_bc', 1.0, 100.0, log=True)
    torch.manual_seed(0)
    return PINN(hidden_layers=layers, hidden_width=width).to(device), lam

best_pinn = _run_study(_build_pinn, name='pinn_search')

# --- SIREN ---
print("\n=== Поиск гиперпараметров SIREN ===")
def _build_siren(trial):
    width   = trial.suggest_categorical('hidden_width',  [32, 64, 128])
    layers  = trial.suggest_int('hidden_layers', 2, 6)
    omega_0 = trial.suggest_int('omega_0', 5, 50)
    lam     = trial.suggest_float('lambda_bc', 1.0, 100.0, log=True)
    torch.manual_seed(0)
    return SIREN(hidden_layers=layers, hidden_width=width, omega_0=omega_0).to(device), lam

best_siren = _run_study(_build_siren, name='siren_search')

# --- SafeNet ---
print("\n=== Поиск гиперпараметров SafeNet ===")
def _build_safenet(trial):
    n_fourier = trial.suggest_categorical('n_fourier', [128, 256, 512])
    sigma     = trial.suggest_float('sigma', 1.0, 20.0)
    layers    = trial.suggest_int('hidden_layers', 1, 3)
    width     = trial.suggest_categorical('hidden_width', [32, 64, 128])
    lam       = trial.suggest_float('lambda_bc', 1.0, 100.0, log=True)
    torch.manual_seed(0)
    return SafeNet(n_fourier=n_fourier, sigma=sigma,
                   hidden_layers=layers, hidden_width=width).to(device), lam

best_safenet = _run_study(_build_safenet, name='safenet_search')

print("\nПоиск завершён")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
torch.manual_seed(42)

# --- PINN ---
_pinn_cfg = {
    'lr':            best_pinn.get('lr',           1e-3),
    'hidden_width':  best_pinn.get('hidden_width',  64),
    'hidden_layers': best_pinn.get('hidden_layers',  4),
    'lambda_bc':     best_pinn.get('lambda_bc',     10.0),
}
model_pinn = PINN(hidden_layers=_pinn_cfg['hidden_layers'],
                  hidden_width=_pinn_cfg['hidden_width']).to(device)
optimizer_pinn = torch.optim.Adam(model_pinn.parameters(), lr=_pinn_cfg['lr'])
print(f"PINN конфиг: {_pinn_cfg}  | параметров: {sum(p.numel() for p in model_pinn.parameters())}")

# --- SIREN ---
_siren_cfg = {
    'lr':            best_siren.get('lr',           5e-5),
    'omega_0':       best_siren.get('omega_0',       10),
    'hidden_width':  best_siren.get('hidden_width',  64),
    'hidden_layers': best_siren.get('hidden_layers',  4),
    'lambda_bc':     best_siren.get('lambda_bc',    10.0),
}
model_siren = SIREN(hidden_layers=_siren_cfg['hidden_layers'],
                    hidden_width=_siren_cfg['hidden_width'],
                    omega_0=_siren_cfg['omega_0']).to(device)
optimizer_siren = torch.optim.Adam(model_siren.parameters(), lr=_siren_cfg['lr'])
print(f"SIREN конфиг: {_siren_cfg}  | параметров: {sum(p.numel() for p in model_siren.parameters())}")

# --- SafeNet ---
_safe_cfg = {
    'lr':            best_safenet.get('lr',           1e-3),
    'n_fourier':     best_safenet.get('n_fourier',    256),
    'sigma':         best_safenet.get('sigma',          5.0),
    'hidden_width':  best_safenet.get('hidden_width',  64),
    'hidden_layers': best_safenet.get('hidden_layers',  1),
    'lambda_bc':     best_safenet.get('lambda_bc',    10.0),
}
model_safe = SafeNet(n_fourier=_safe_cfg['n_fourier'], sigma=_safe_cfg['sigma'],
                     hidden_layers=_safe_cfg['hidden_layers'],
                     hidden_width=_safe_cfg['hidden_width']).to(device)
optimizer_safe = torch.optim.Adam(model_safe.parameters(), lr=_safe_cfg['lr'])
print(f"SafeNet конфиг: {_safe_cfg}  | параметров: {sum(p.numel() for p in model_safe.parameters())}")

In [ ]:
NTK_UPDATE = 500
N_col      = 2000
N_bc       = 500
n_epochs   = 10000
log_every  = 1000


def compute_ntk_lambda(model, xy_col, xy_bc):
    lp = pde_loss(model, xy_col)
    gp = torch.autograd.grad(lp, model.parameters(), allow_unused=True)
    norm_pde = sum(g.detach().norm()**2 for g in gp if g is not None).sqrt().item()
    lb = bc_loss(model, xy_bc)
    gb = torch.autograd.grad(lb, model.parameters(), allow_unused=True)
    norm_bc = sum(g.detach().norm()**2 for g in gb if g is not None).sqrt().item()
    return float(np.clip(norm_pde / (norm_bc + 1e-8), 0.1, 1000.0))


xy_col_train = torch.rand(N_col, 2, device=device, requires_grad=True)
xy_bc_train  = sample_boundary(N_bc).to(device)

lam_p = float(_pinn_cfg['lambda_bc'])
lam_s = float(_siren_cfg['lambda_bc'])
lam_f = float(_safe_cfg['lambda_bc'])

history_pinn  = {'pde': [], 'bc': [], 'total': [], 'lambda': []}
history_siren = {'pde': [], 'bc': [], 'total': [], 'lambda': []}
history_safe  = {'pde': [], 'bc': [], 'total': [], 'lambda': []}

for epoch in range(1, n_epochs + 1):
    if epoch % NTK_UPDATE == 1:
        lam_p = compute_ntk_lambda(model_pinn,  xy_col_train, xy_bc_train)
        lam_s = compute_ntk_lambda(model_siren, xy_col_train, xy_bc_train)
        lam_f = compute_ntk_lambda(model_safe,  xy_col_train, xy_bc_train)

    xy_col_train.grad = None

    for model, opt, lam, hist in [
        (model_pinn,  optimizer_pinn,  lam_p, history_pinn),
        (model_siren, optimizer_siren, lam_s, history_siren),
        (model_safe,  optimizer_safe,  lam_f, history_safe),
    ]:
        opt.zero_grad()
        lp = pde_loss(model, xy_col_train)
        lb = bc_loss(model, xy_bc_train)
        loss = lp + lam * lb
        loss.backward()
        opt.step()
        hist['pde'].append(lp.item())
        hist['bc'].append(lb.item())
        hist['total'].append(loss.item())
        hist['lambda'].append(lam)

    if epoch % log_every == 0:
        print(f'Эпоха {epoch:5d} | '
              f'PINN: {history_pinn["total"][-1]:.4e} (λ={lam_p:.1f}) | '
              f'SIREN: {history_siren["total"][-1]:.4e} (λ={lam_s:.1f}) | '
              f'SafeNet: {history_safe["total"][-1]:.4e} (λ={lam_f:.1f})')

print('Обучение завершено')

In [ ]:
def moving_avg(x, w=200):
    return np.convolve(x, np.ones(w) / w, mode='valid')

epochs_full = np.arange(1, n_epochs + 1)
epochs_ma   = np.arange(w := 200, n_epochs + 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
titles = ['Суммарные потери', 'PDE потери', 'BC потери']
keys   = ['total', 'pde', 'bc']
colors = {'pinn': 'steelblue', 'siren': 'darkorange', 'safe': 'forestgreen'}
hists  = {'pinn': history_pinn, 'siren': history_siren, 'safe': history_safe}
labels = {'pinn': 'PINN (tanh)', 'siren': 'SIREN (sin)', 'safe': 'SafeNet'}

for ax, title, key in zip(axes[:3], titles, keys):
    for name, hist in hists.items():
        ax.semilogy(epochs_full, hist[key], color=colors[name], alpha=0.15)
        ax.semilogy(epochs_ma, moving_avg(hist[key]), color=colors[name], label=labels[name])
    ax.set_title(title)
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('Потери (лог. шкала)')
    ax.legend()
    ax.grid(True, which='both', ls='--')

for name, hist in hists.items():
    axes[3].plot(epochs_full, hist['lambda'], color=colors[name], label=labels[name])
axes[3].set_title('NTK λ')
axes[3].set_xlabel('Эпоха')
axes[3].set_ylabel('λ')
axes[3].legend()
axes[3].grid(True, which='both', ls='--')

plt.tight_layout()
plt.show()

In [ ]:
for m in (model_pinn, model_siren, model_safe):
    m.eval()

N_vis = 100
x_lin = torch.linspace(0, 1, N_vis)
y_lin = torch.linspace(0, 1, N_vis)
XX, YY = torch.meshgrid(x_lin, y_lin, indexing='ij')
xy_vis = torch.stack([XX.flatten(), YY.flatten()], dim=1).to(device)

with torch.no_grad():
    u_true        = u_exact(xy_vis[:, 0], xy_vis[:, 1]).reshape(N_vis, N_vis).cpu().numpy()
    u_pred_pinn   = model_pinn(xy_vis).squeeze().reshape(N_vis, N_vis).cpu().numpy()
    u_pred_siren  = model_siren(xy_vis).squeeze().reshape(N_vis, N_vis).cpu().numpy()
    u_pred_safe   = model_safe(xy_vis).squeeze().reshape(N_vis, N_vis).cpu().numpy()

err_pinn  = np.abs(u_pred_pinn  - u_true)
err_siren = np.abs(u_pred_siren - u_true)
err_safe  = np.abs(u_pred_safe  - u_true)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

row0 = [(u_true, 'Точное решение'), (u_pred_pinn,  'PINN предсказание'),
        (u_pred_siren, 'SIREN предсказание'), (u_pred_safe, 'SafeNet предсказание')]
row1 = [(err_pinn,  '|Ошибка| PINN'), (err_siren, '|Ошибка| SIREN'),
        (err_safe, '|Ошибка| SafeNet'), (None, None)]

for col, (data, title) in enumerate(row0):
    im = axes[0, col].imshow(data.T, origin='lower', extent=[0,1,0,1], cmap='viridis', aspect='equal')
    axes[0, col].set_title(title)
    axes[0, col].set_xlabel('x'); axes[0, col].set_ylabel('y')
    plt.colorbar(im, ax=axes[0, col])

for col, (data, title) in enumerate(row1):
    if data is None:
        axes[1, col].axis('off')
        continue
    im = axes[1, col].imshow(data.T, origin='lower', extent=[0,1,0,1], cmap='hot_r', aspect='equal')
    axes[1, col].set_title(title)
    axes[1, col].set_xlabel('x'); axes[1, col].set_ylabel('y')
    plt.colorbar(im, ax=axes[1, col])

plt.tight_layout()
plt.show()

In [ ]:
def relative_l2(u_pred, u_true):
    return np.linalg.norm(u_pred - u_true) / np.linalg.norm(u_true)

results = {
    'PINN (tanh)':  relative_l2(u_pred_pinn,  u_true),
    'SIREN (sin)':  relative_l2(u_pred_siren, u_true),
    'SafeNet':      relative_l2(u_pred_safe,  u_true),
}
best_model = min(results, key=results.get)

df = pd.DataFrame({
    'Модель':                  list(results.keys()),
    'Относительная ошибка L2': [f'{v:.4e}' for v in results.values()],
    'Лучше':                   ['✓' if k == best_model else '' for k in results],
})
print(df.to_string(index=False))